# B2 · Ảnh + chữ → vector số · làm sạch tự động

Mỗi ảnh / mỗi câu được mô hình có sẵn đổi thành một dãy số (vector). Mọi bước sau đều dùng vector này.
GPU: 10.000 ảnh khoảng 1–3 phút mỗi mô hình. Kết quả được lưu đệm, chạy lại không tốn thời gian.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # thư mục notebooks (chứa b2lib.py)
import numpy as np, pandas as pd
import b2lib as L
print("thư mục làm việc:", L.WORK)

In [ ]:
df = L.doc_bang_chuan()
itr, ite = (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
y = df.loc[itr, "label"].astype(str).to_numpy()
ids_tr, ids_te = df.loc[itr, "id"].to_numpy(), df.loc[ite, "id"].to_numpy()
print(itr.sum(), "train |", ite.sum(), "test |", len(set(y)), "lớp")

## 1. ✏️ Chọn mô hình
Ảnh: `clip` (nhanh, hiểu nội dung chung), `dinov2` (chi tiết hình dạng), `siglip`, `clip_l` (lớn, tốt hơn, chậm hơn).
Chữ: `e5` (đa ngôn ngữ, nhanh), `e5_b` (lớn hơn), `bkai` (chuyên tiếng Việt), `clip_text` (cùng không gian với ảnh CLIP — dùng cho bài ghép ảnh–chữ).

In [ ]:
F = L.dac_trung(df, anh=("clip", "dinov2"), chu=("e5", "bkai"))
np.savez(L.WORK / "dac_trung.npz", **F)
{k: v.shape for k, v in F.items()}

## 2. Làm sạch tự động: tìm dòng nghi SAI NHÃN và bỏ ra (tối đa 10%)

Máy chấm chéo 5 phần trên vector: dòng nào mô hình (chưa từng thấy dòng đó) rất chắc là nhãn khác thì nghi sai.
Danh sách: `work_b2/nghi_sai_nhan.csv`. Bài đoán số thì bỏ qua ô này.

In [ ]:
giu, rep = L.lam_sach_tu_dong(np.hstack([F[k][itr] for k in F]), y, ids_tr)
np.save(L.WORK / "giu_lai.npy", giu)
rep.head(20)